# F12 - Tensors, gradients, loss functions, optimizers, and backpropagation

**Hardware:** CPU-only; optional GPU · **Time:** 4 hours · **Prerequisite:** F11

This notebook uses PyTorch. You will inspect tensor shapes, compute gradients
with `autograd`, verify a gradient numerically, train a small multilayer network,
and plot training and validation curves. Every computation runs on CPU. The
device line will use CUDA only if it is available.

In [1]:
from pathlib import Path
import platform
import random

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
print("torch", torch.__version__, "| device", DEVICE, "| cuda", torch.cuda.is_available())
assert torch.__version__.startswith("2.")

torch 2.4.1+cpu | device cpu | cuda False


## 1. Tensors: shape, dtype, and broadcasting

A tensor is an n-dimensional array with a device and a dtype. Matrix
multiplication uses the last two dimensions. Broadcasting expands compatible
dimensions; it is powerful and a common source of silent shape bugs. Print shapes
before and after every operation that changes them.

In [2]:
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
b = torch.tensor([10.0, 20.0])
print("a:", a.shape, "| b:", b.shape)
print("a + b:", a + b)
print("a @ b:", a @ b)
assert (a + b).shape == (2, 2)
assert (a @ b).shape == (2,)

batch = torch.randn(8, 3)
weights = torch.randn(3, 4)
output = batch @ weights
print("batch:", batch.shape, "weights:", weights.shape, "output:", output.shape)
assert output.shape == (8, 4)

a: torch.Size([2, 2]) | b: torch.Size([2])
a + b: tensor([[11., 22.],
        [13., 24.]])
a @ b: tensor([ 50., 110.])
batch: torch.Size([8, 3]) weights: torch.Size([3, 4]) output: torch.Size([8, 4])


## 2. Automatic differentiation

PyTorch records operations on tensors with `requires_grad=True`. Calling
`backward()` applies the chain rule and stores derivatives in `.grad`. For
`y = x^2 + 3x`, the derivative is `2x + 3`, which is 7 at `x = 2`.

In [3]:
x = torch.tensor(2.0, requires_grad=True)
y = x**2 + 3 * x
y.backward()
print("dy/dx at x=2:", x.grad.item())
assert x.grad.item() == 7.0

dy/dx at x=2: 7.0


## 3. Verify a gradient numerically

Autograd is convenient but not magic. A finite-difference check compares the
analytic gradient with a small perturbation. The two should agree to a few
decimal places. This is the same check you would use when implementing a custom
layer.

In [4]:
def loss_value(w_value: float, b_value: float) -> float:
    x_data = torch.linspace(-1, 1, 50)
    target = 2 * x_data + 1
    prediction = w_value * x_data + b_value
    return float(((prediction - target) ** 2).mean())


w = torch.tensor(1.5, requires_grad=True)
b = torch.tensor(0.2, requires_grad=True)
x_data = torch.linspace(-1, 1, 50)
target = 2 * x_data + 1
loss = ((w * x_data + b - target) ** 2).mean()
loss.backward()

epsilon = 1e-3
numeric = (loss_value(1.5 + epsilon, 0.2) - loss_value(1.5 - epsilon, 0.2)) / (2 * epsilon)
print("autograd dL/dw:", round(w.grad.item(), 6))
print("numeric dL/dw:", round(numeric, 6))
assert abs(w.grad.item() - numeric) < 1e-2

autograd dL/dw: -0.346939
numeric dL/dw: -0.346988


## 4. A small model, loss, and optimizer

The model has two inputs, two hidden layers, and one output logit.
`BCEWithLogitsLoss` combines a sigmoid with binary cross-entropy and is
numerically more stable than applying sigmoid and then `BCELoss`. Adam adapts a
separate step size per parameter.

In [5]:
X, y = make_moons(n_samples=800, noise=0.25, random_state=SEED)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train).astype("float32")
X_test = scaler.transform(X_test).astype("float32")

X_train_t = torch.tensor(X_train, dtype=torch.float32, device=DEVICE)
y_train_t = torch.tensor(y_train, dtype=torch.float32, device=DEVICE).view(-1, 1)
X_test_t = torch.tensor(X_test, dtype=torch.float32, device=DEVICE)
y_test_t = torch.tensor(y_test, dtype=torch.float32, device=DEVICE).view(-1, 1)
print("train tensors:", X_train_t.shape, y_train_t.shape)
assert X_train_t.shape[0] == y_train_t.shape[0]

train tensors: torch.Size([600, 2]) torch.Size([600, 1])


In [6]:
model = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(model)
print("parameters:", parameter_count)
assert parameter_count > 0

Sequential(
  (0): Linear(in_features=2, out_features=16, bias=True)
  (1): ReLU()
  (2): Linear(in_features=16, out_features=16, bias=True)
  (3): ReLU()
  (4): Linear(in_features=16, out_features=1, bias=True)
)
parameters: 337


## 5. Train and watch two curves

Training loss alone can keep falling while validation performance gets worse.
Track both. The gap between them is the overfitting signal; a high value on both
is underfitting.

In [7]:
def evaluate() -> tuple[float, float]:
    model.eval()
    with torch.no_grad():
        train_logits = model(X_train_t)
        val_logits = model(X_test_t)
        train_loss = criterion(train_logits, y_train_t).item()
        val_loss = criterion(val_logits, y_test_t).item()
        train_acc = ((train_logits > 0).float() == y_train_t).float().mean().item()
        val_acc = ((val_logits > 0).float() == y_test_t).float().mean().item()
    return train_loss, val_loss, train_acc, val_acc


history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
for epoch in range(1, 201):
    model.train()
    optimizer.zero_grad()
    logits = model(X_train_t)
    loss = criterion(logits, y_train_t)
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0 or epoch == 1:
        train_loss, val_loss, train_acc, val_acc = evaluate()
        for key, value in zip(history, (train_loss, val_loss, train_acc, val_acc)):
            history[key].append(value)
        if epoch % 50 == 0:
            print(f"epoch {epoch:3d} | train loss {train_loss:.3f} | val loss {val_loss:.3f} | val acc {val_acc:.3f}")

train_loss, val_loss, train_acc, val_acc = evaluate()
print("final train accuracy:", round(train_acc, 3))
print("final validation accuracy:", round(val_acc, 3))
assert val_acc > 0.90

epoch  50 | train loss 0.146 | val loss 0.153 | val acc 0.950
epoch 100 | train loss 0.128 | val loss 0.161 | val acc 0.945
epoch 150 | train loss 0.124 | val loss 0.173 | val acc 0.945
epoch 200 | train loss 0.121 | val loss 0.177 | val acc 0.945
final train accuracy: 0.948
final validation accuracy: 0.945


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("checkpoint")
axes[0].legend()
axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"], label="validation")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("checkpoint")
axes[1].legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "f12_training_curves.png", bbox_inches="tight")
plt.close(fig)
assert (OUTPUT_DIR / "f12_training_curves.png").exists()

## 6. Backpropagation in one paragraph

For each layer, the forward pass stores the input and output. The loss produces a
gradient with respect to the output. The chain rule propagates that gradient
backward through each layer: multiply by the local derivative, accumulate the
parameter gradient, and pass the input gradient to the previous layer. The
optimizer then updates each parameter in the opposite direction of its gradient.

| Quantity | Shape | Meaning |
| --- | --- | --- |
| `X` | `(n, 2)` | input batch |
| `W1` | `(2, 16)` | first-layer weights |
| `H1` | `(n, 16)` | hidden activations |
| `W3` | `(16, 1)` | output weights |
| `loss` | scalar | mean binary cross-entropy |

**Next:** F13 - CPU/GPU basics, random seeds, reproducibility, and experiment tracking.